# 01 Naive / Seasonal Naive Forecast

F1で作った `src/forecasting/` の関数を使い、固定分割A/Bで naive forecast と seasonal naive forecast を実行する。予測結果、評価指標、図はすべて `output/forecasts/` 配下に保存する。

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display


PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT.name != "Transport_amount_project" and PROJECT_ROOT.parent != PROJECT_ROOT:
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data_loader import load_connected_parcel_data
from src.forecasting.evaluation import evaluate_forecasts
from src.forecasting.naive import naive_forecast, seasonal_naive_forecast
from src.forecasting.splits import make_fixed_split_a, make_fixed_split_b


DATA_PATH = PROJECT_ROOT / "data" / "processed" / "parcel_volume_connected.csv"
FORECAST_DIR = PROJECT_ROOT / "output" / "forecasts"
PREDICTIONS_DIR = FORECAST_DIR / "predictions"
METRICS_DIR = FORECAST_DIR / "metrics"
FIGURES_DIR = FORECAST_DIR / "figures"

for path in [PREDICTIONS_DIR, METRICS_DIR, FIGURES_DIR]:
    path.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Data path:", DATA_PATH)
print("Forecast output:", FORECAST_DIR)

## 1. データ読み込みと固定分割

予測評価は原系列スケールの `number_parcels` で行う。固定分割Bはデータ期間が不足する環境では skip できるようにする。

In [ ]:
df = load_connected_parcel_data(str(DATA_PATH))

splits = {}
split_rows = []
for split_name, maker in [("fixed_a", make_fixed_split_a), ("fixed_b", make_fixed_split_b)]:
    try:
        split = maker(df)
        splits[split_name] = split
        split_rows.append(
            {
                "split": split_name,
                "status": "success",
                "train_start": split["train"].index.min().date(),
                "train_end": split["train"].index.max().date(),
                "train_rows": len(split["train"]),
                "test_start": split["test"].index.min().date(),
                "test_end": split["test"].index.max().date(),
                "test_rows": len(split["test"]),
                "message": "",
            }
        )
    except ValueError as exc:
        split_rows.append(
            {
                "split": split_name,
                "status": "skipped",
                "train_start": None,
                "train_end": None,
                "train_rows": 0,
                "test_start": None,
                "test_end": None,
                "test_rows": 0,
                "message": str(exc),
            }
        )

split_summary = pd.DataFrame(split_rows)
display(split_summary)

## 2. Naive / Seasonal Naive の実行

`naive_forecast` は学習期間の最後の値をテスト期間すべてに使う。`seasonal_naive_forecast` は前年同月を使う。固定分割の unconditional forecast では、テスト期間内の実績値を予測に使わない。テスト期間が12か月を超える場合、前年同月がテスト期間内に入るが、その場合は過去に生成した予測値を再帰的に使う。

In [ ]:
prediction_frames = []
metrics_frames = []

for split_name, split in splits.items():
    spec_name = "baseline_m4" if split_name == "fixed_a" else "post2020_m5"

    naive_df = naive_forecast(
        split["train"],
        split["test"],
        split=split_name,
        spec_name=spec_name,
    )
    seasonal_df = seasonal_naive_forecast(
        split["train"],
        split["test"],
        split=split_name,
        spec_name=spec_name,
    )

    split_predictions = pd.concat([naive_df, seasonal_df], ignore_index=True)
    prediction_frames.append(split_predictions)

    split_metrics = evaluate_forecasts(
        split_predictions,
        y_train=split["train"]["number_parcels"],
    )
    metrics_frames.append(split_metrics)

predictions_df = pd.concat(prediction_frames, ignore_index=True) if prediction_frames else pd.DataFrame()
metrics_df = pd.concat(metrics_frames, ignore_index=True) if metrics_frames else pd.DataFrame()

display(predictions_df.head())
display(metrics_df)

## 3. Seasonal naive がテスト実績を使っていないことの確認

固定分割Aのテスト期間は24か月なので、13か月目以降の前年同月はテスト期間内に入る。この実装では、その月の実績値ではなく、すでに生成した予測値を履歴に入れて再帰的に使う。下の確認では、2021年1月の seasonal naive 予測が 2020年1月の実績値ではなく、2020年1月に生成した予測値と一致することを見る。

In [ ]:
check_rows = []
if "fixed_a" in splits:
    fixed_a_seasonal = predictions_df[(predictions_df["split"] == "fixed_a") & (predictions_df["model"] == "seasonal_naive")].copy()
    pred_2020_01 = fixed_a_seasonal.loc[fixed_a_seasonal["date"] == pd.Timestamp("2020-01-01"), "y_pred"].iloc[0]
    pred_2021_01 = fixed_a_seasonal.loc[fixed_a_seasonal["date"] == pd.Timestamp("2021-01-01"), "y_pred"].iloc[0]
    actual_2020_01 = splits["fixed_a"]["test"].loc[pd.Timestamp("2020-01-01"), "number_parcels"]
    check_rows.append(
        {
            "check": "2021-01 uses generated 2020-01 prediction",
            "pred_2020_01": pred_2020_01,
            "actual_2020_01": actual_2020_01,
            "pred_2021_01": pred_2021_01,
            "matches_prior_prediction": bool(pred_2021_01 == pred_2020_01),
            "uses_test_actual": bool(pred_2021_01 == actual_2020_01),
        }
    )

display(pd.DataFrame(check_rows))

## 4. 予測結果と評価指標の保存

分割ごとの予測CSV、全体の評価指標CSVを `output/forecasts/` 配下に保存する。

In [ ]:
if "fixed_a" in splits:
    predictions_df[predictions_df["split"] == "fixed_a"].to_csv(PREDICTIONS_DIR / "fixed_a_naive.csv", index=False)

if "fixed_b" in splits:
    predictions_df[predictions_df["split"] == "fixed_b"].to_csv(PREDICTIONS_DIR / "fixed_b_naive.csv", index=False)

metrics_df.to_csv(METRICS_DIR / "naive_metrics.csv", index=False)

print("Saved prediction and metric CSV files.")

## 5. 予測図の保存

各固定分割について、テスト期間の実績値と naive / seasonal naive 予測を比較する。学習期間の最後の24か月も薄く表示して、テスト期間とのつながりを見やすくする。

In [ ]:
def plot_split_forecasts(split_name: str, save_path: Path) -> None:
    split = splits[split_name]
    plot_df = predictions_df[predictions_df["split"] == split_name]

    fig, ax = plt.subplots(figsize=(10.5, 5.5))
    train_tail = split["train"].tail(24)
    ax.plot(train_tail.index, train_tail["number_parcels"], color="0.55", linewidth=1.2, label="train actual tail")
    ax.plot(split["test"].index, split["test"]["number_parcels"], color="black", linewidth=1.6, label="test actual")

    for model_name, group in plot_df.groupby("model"):
        ax.plot(group["date"], group["y_pred"], marker="o", linewidth=1.2, label=model_name)

    ax.axvline(split["train"].index.max(), color="0.2", linestyle=":", linewidth=1.0)
    ax.set_title(f"{split_name}: naive forecast comparison")
    ax.set_xlabel("Date")
    ax.set_ylabel("number_parcels")
    ax.grid(True, color="0.85", linewidth=0.8)
    ax.legend()
    fig.tight_layout()
    fig.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.close(fig)


if "fixed_a" in splits:
    plot_split_forecasts("fixed_a", FIGURES_DIR / "fixed_a_naive_forecast.png")

if "fixed_b" in splits:
    plot_split_forecasts("fixed_b", FIGURES_DIR / "fixed_b_naive_forecast.png")

print("Saved forecast figures.")